In [ ]:
from pathlib import Path

import pandas as pd
import torch
from tqdm import tqdm
from transformers import MarianMTModel, MarianTokenizer

In [ ]:


# ===== 1) Đọc file =====
APP_DIR = Path.cwd()
if not (APP_DIR / "Dataset").exists() and (APP_DIR / "hfgat_rewrite_validate" / "Dataset").exists():
    APP_DIR = APP_DIR / "hfgat_rewrite_validate"
DATA_ROOT = APP_DIR / "Dataset"

df = pd.read_csv(DATA_ROOT / "item_data.txt", header=None)
df.columns = ["col0", "col1", "image_url", "title"]
df["title"] = df["title"].fillna("").astype(str)

# ===== 2) Chỉ lấy title unique =====
unique_titles = [t.strip() for t in df["title"].unique() if t.strip() != ""]

# ===== 3) Load model dịch zh -> en =====
model_name = "Helsinki-NLP/opus-mt-zh-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

# ===== 4) Hàm dịch batch =====
def translate_batch(texts, max_length=64):
    enc = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    
    with torch.no_grad():
        generated = model.generate(
            **enc,
            max_length=max_length,
            num_beams=4
        )
    return tokenizer.batch_decode(generated, skip_special_tokens=True)

# ===== 5) Dịch toàn bộ unique_titles theo batch =====
mapping = {}
batch_size = 128 if device == "cuda" else 32

for i in tqdm(range(0, len(unique_titles), batch_size)):
    batch = unique_titles[i:i+batch_size]
    outs = translate_batch(batch)
    for src, tgt in zip(batch, outs):
        mapping[src] = tgt.strip()

# ===== 6) Map lại vào dataframe =====
df["title_en"] = df["title"].astype(str).str.strip().map(mapping).fillna("")

# ===== 7) Lưu =====
df.to_csv(DATA_ROOT / "item_data_translated.csv", index=False)
print("Done")